In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

d = fetch_ucirepo(id=967)  # download do dataset
df = pd.concat([d.data.features, d.data.targets], axis=1)   # tabela com caracteristicas do url e target

df["phishing"] = 1 - df["label"]        # no dataset original 1 = legítimo, 0 = phishing; inversão para ficar intuitivo

df.to_csv("../data/phiusiil.csv", index=False)
print(df.shape)                         # numero linhas e colunas
df["phishing"].value_counts()           # quais são phishing (1) e quais não (0)

(235795, 56)


phishing
0    134850
1    100945
Name: count, dtype: int64

In [3]:
df[["URL", "URLLength", "NoOfSubDomain", "IsHTTPS", "phishing"]].sample(10)

,URL,URLLength,NoOfSubDomain,IsHTTPS,phishing
145084,https://www.climatologylab.org,29,1,1,0
4545,http://www.handyman.macleannsw.com,33,2,0,1
21186,https://unirelomoving.com/login/ourtime/,40,0,1,1
48891,http://www.sahityanepal.com,26,1,0,1
64961,http://mr-robot-101.github.io/facebook-log-in-...,57,1,0,1
9522,http://www.medittech.com,23,1,0,1
42355,https://www.logoverge.com,24,1,1,0
98837,https://www.electrisave.co.uk,28,2,1,0
216475,https://www.polyfriendly.org,27,1,1,0
106918,https://www.motherteresawomenuniv.ac.in,38,2,1,0


In [ ]:
from urllib.parse import urlparse
import math
from collections import Counter

PALAVRAS = ["verify", "secure", "account", "update", "bank"]

MARCAS = {
    "facebook": "facebook.com",
    "instagram": "instagram.com",
    "linkedin": "linkedin.com",
    "dhl": "dhl.com",
    "docusign": "docusign.com",
    "dropbox": "dropbox.com",
    "ctt": "ctt.pt",
    "santander": "santander.pt",
    "millenniumbcp": "millenniumbcp.pt",
}

def marca_falsa(url, host):
    host = host.split("@")[-1].split(":")[0].lower()
    for marca, dominio in MARCAS.items():
        if marca in url.lower():
            if host != dominio and not host.endswith("." + dominio):
                return 1
    return 0


def entropia(s):
    # mede o "quão aleatório" é o texto (domínios gerados por máquina têm valores altos)
    if not s:
        return 0
    c = Counter(s)
    return -sum(n / len(s) * math.log2(n / len(s)) for n in c.values())  # entropia de Shannon



def extrair_features(url):
    p = urlparse(url)
    host = p.netloc  # ex.: "paypa1-login.xyz"
    return {
        "comprimento": len(url),
        "n_pontos": host.count("."),
        "n_hifenes": host.count("-"),
        "n_digitos": sum(c.isdigit() for c in url),
        "tem_https": int(p.scheme == "https"),
        "tem_arroba": int("@" in url),
        "tem_ip": int(host.replace(".", "").isdigit()),
        "tem_login": int("login" in url.lower()),
        "entropia": entropia(host),
        "n_palavras_suspeitas": sum(w in url.lower() for w in PALAVRAS),
        "tem porta": int(":" in host.split("@")[-1]),
        "marca_falsa": marca_falsa(url, host)

    }

extrair_features("https://paypa1-login-secure.xyz/verify")


{'comprimento': 38,
 'n_pontos': 1,
 'n_hifenes': 2,
 'n_digitos': 1,
 'tem_https': 1,
 'tem_arroba': 0,
 'tem_ip': 0,
 'tem_login': 1,
 'entropia': 4.088779347361361,
 'n_palavras_suspeitas': 2,
 'tem porta': 0}

In [11]:
X = pd.DataFrame([extrair_features(u) for u in df["URL"]])
y = df["phishing"]
X.head()

,comprimento,n_pontos,n_hifenes,n_digitos,tem_https,tem_arroba,tem_ip,tem_login,entropia
0,32,2,0,0,1,0,0,0,3.657268
1,24,2,1,0,1,0,0,0,3.327820
2,30,3,0,0,1,0,0,0,3.629220
3,27,2,0,0,1,0,0,0,3.576618
4,34,2,0,0,1,0,0,0,3.430457


In [14]:
from sklearn.model_selection import train_test_split                        # divide os dados em treino e teste
from xgboost import XGBClassifier                                           # modelo classificador
from sklearn.metrics import classification_report, confusion_matrix         # medidor qualidade


# etapa 1 - Dividir os Dados
# X = as pistas (features), y = a resposta certa (0 legítimo, 1 phishing)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,               # o que vamos dividir
    test_size=0.2,      # 20% vão para teste, 80% para treino
    stratify=y, 
    random_state=42
)

# X_train/y_train: o que o modelo vai estudar
# X_test/y_test:   o que ele nunca viu (teste surpresa)


# 2. treinar
model = XGBClassifier(n_estimators=200, max_depth=6, random_state=42)   # nº de árvores de decisão, profundidade máxima cada, resultados repetiveis
model.fit(X_train, y_train)              # o modelo estuda: aprende que pistas levam a que resposta

# 3. testar
pred = model.predict(X_test)
print(classification_report(y_test, pred, target_names=["legítimo", "phishing"]))     # compara as respostas dele (pred) com as certas (y_test)
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

    legítimo       0.97      0.99      0.98     26970
    phishing       0.98      0.95      0.97     20189

    accuracy                           0.97     47159
   macro avg       0.98      0.97      0.97     47159
weighted avg       0.97      0.97      0.97     47159

[[26665   305]
 [  935 19254]]


In [15]:
pd.Series(
    model.feature_importances_,   # quanto cada pista pesou nas decisões do modelo
    index=X.columns               # associa cada número ao nome da pista
).sort_values(ascending=False)    # ordena da mais importante para a menos

tem_https      0.900394
n_digitos      0.063295
n_pontos       0.017438
comprimento    0.007308
tem_login      0.006404
n_hifenes      0.003408
entropia       0.001282
tem_arroba     0.000471
tem_ip         0.000000
dtype: float32